# 003b - Local Indexing batch

Written by Jean-Baptiste Jacob

Last updated: 18/02/2026

Run local_indexing on a series of datasets, using options fine-tuned in fp3 test notebook. Sends one slurm job per dataset. 
This works only for indexing a single crystal structure acrosss all dataset. For polymineralic samples, re-run this for each phase to index.

In [ ]:
import sys

# python environment stuff
IMAGED11_PATH = '/home/esrf/jean1994b/ImageD11_jbjacob'  # None means do not use git, otherwise enter the name of the folder to use for the git checkout "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = 'ImageD11'  # the name of the git checkout folder within path. None means guess

if IMAGED11_PATH is not None:
    if '/data/id11/nanoscope' not in sys.path:
        sys.path.append('/data/id11/nanoscope')
    import install_ImageD11_from_git
    PYTHONPATH = install_ImageD11_from_git.setup_ImageD11_from_git(IMAGED11_PATH,CHECKOUT_PATH)
else:
    import site
    PYTHONPATH = site.getsitepackages()[0]
    print(PYTHONPATH)

In [ ]:
import os
import matplotlib.pyplot as pl
import numpy as np

import ImageD11.sinograms.dataset, ImageD11.columnfile
    
from pf3dxrd.pf3dxrd import utils, pixelmap, local_indexing

%matplotlib ipympl
%load_ext autoreload
%autoreload 2

In [ ]:
root = '/home/esrf/jean1994b/ES1832/PROCESSED_DATA/'    # root directory of your project
phase = 'MgO'                                           # phase to index
parfile = '../tdxrd_pars/pars.json'                     # parfile to read calibration geometry

In [ ]:
# sample list. guess it from folders in root or define manually
skip=['json', 'ipynb']
samples = [p for p in os.listdir(root) if ('MgO' in p) and all([sk not in p for sk in skip])]
samples

In [ ]:
# datasets
skip = ['ipynb', 'match','pct', 'pycache','json', 'svg', 'cif', 'combined']  # skip these

process_dict = {}
for s in samples:
    sample_dir = os.path.join(root,s)
    dsets = sorted([ds for ds in os.listdir(sample_dir) if all([sk not in ds for sk in skip])])
    process_dict[s] = dsets

process_dict

In [ ]:
# load indexing parameters from json file
indexing_pars = os.path.join(root, samples[-1], f'indexing_pars_{phase}.json')

#### Check indexing options
have a look at indexing options and update them if needed. If 003_local_indexing_test was run beforehand, they should be saved in a json file. Otherwise, they can be defined and saved here.
OPTS.unitcell is not saved but re-defined directly in the main() function

In [ ]:
OPTS = local_indexing.Options.load(indexing_pars)
#OPTS.save(indexing_pars)
print(OPTS)

In [ ]:
# unitcell has not been initialized. read it from one xmap
sample = samples[0]
dset = process_dict[sample][0]
xmapfile =  os.path.join(root,f'{sample}',f'{dset}',f'{dset}_xmap.h5')
xmap = pixelmap.load_from_hdf5(xmapfile)
cs = xmap.phases.get(phase)
print(cs)

In [ ]:
# add it to opts
OPTS.unitcell = cs.to_ImageD11_unitcell()

In [ ]:
# don't forget to save
OPTS.save(indexing_pars)

### prepare batch command and run

In [ ]:
def get_dsfile(sample, dset):
    return os.path.join(root,f'{sample}', f'{dset}',f'{dset}_dataset.h5')

def get_peakfile(sample, dset):
    return os.path.join(root,f'{sample}', f'{dset}', f'{dset}_peaks_2d_paired.h5')

def get_command(sample, dset):
    return f'{local_indexing.__file__} -dsfile {get_dsfile(sample, dset)} -pksfile {get_peakfile(sample, dset)} -pname {phase} -indexing_pars {indexing_pars} -usecluster True'

In [ ]:
command_queue = []
for spl in process_dict:
    for dset in process_dict[spl]:
        command_queue.append(get_command(spl, dset))
command_queue[1]

In [ ]:
!python {command_queue[1]}

In [ ]:
# submit
for command in command_queue:
    !python {command}

In [ ]:
!squeue --me